In [43]:
# Hospital Readmission Prediction
# Logistic Regression with L2 Regularization

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

In [44]:
# 1. Load the original dataset

data = pd.read_csv("/data/casestudy-1-hospital data analysis.csv")

# Clean column names
data.columns = data.columns.str.strip()

print("Dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

print("\nFirst 5 rows:")
print(data.head())

Dataset shape: (984, 10)

Columns:

['Patient_ID', 'Age', 'Gender', 'Condition', 'Procedure', 'Cost', 'Length_of_Stay', 'Readmission', 'Outcome', 'Satisfaction']

First 5 rows:

   Patient_ID  Age  Gender  ... Readmission    Outcome  Satisfaction

...

1           2   60    Male  ...         Yes     Stable             3

2           3   32  Female  ...          No  Recovered             5

3           4   75    Male  ...         Yes     Stable             2

4           5   50  Female  ...          No  Recovered             4

[5 rows x 10 columns]

In [45]:
# 2. Remove Patient_ID

data = data.drop(columns=["Patient_ID"], errors="ignore")

In [46]:
# 3. Convert Readmission into 0 and 1

data["Readmission"] = data["Readmission"].astype(str).str.strip()

data["Readmission"] = data["Readmission"].map({
    "No": 0,
    "Yes": 1
})

# Check for missing target values
print("\nReadmission values:")
print(data["Readmission"].value_counts(dropna=False))

# Remove rows where Readmission is missing
data = data.dropna(subset=["Readmission"])

# Convert to integer
data["Readmission"] = data["Readmission"].astype(int)

Readmission values:

0    720

1    264

Name: Readmission, dtype: int64

In [47]:
# 4. Separate input and output

X = data.drop(columns=["Readmission"])
y = data["Readmission"]

In [48]:
# 5. Numerical and categorical columns

numerical_columns = [
    "Age",
    "Cost",
    "Length_of_Stay",
    "Satisfaction"
]

categorical_columns = [
    "Gender",
    "Condition",
    "Procedure",
    "Outcome"
]

In [49]:
# 6. Preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), numerical_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         categorical_columns)
    ]
)


In [50]:
# 7. Logistic Regression with L2 Regularization

model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    class_weight="balanced"
)

In [51]:
# 8. Create pipeline

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [52]:
# 9. Split data into training and testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (787, 8)

Testing data: (197, 8)

In [53]:
# 10. Train the model

pipeline.fit(X_train, y_train)

print("\nModel trained successfully!")

Model trained successfully!

In [54]:
# 10b. Tune L2 regularization strength (C):

from sklearn.model_selection import GridSearchCV

param_grid = {"model__C": [0.01, 0.1, 0.5, 1, 5, 10]}
grid = GridSearchCV(pipeline, param_grid, scoring="roc_auc", cv=5)
grid.fit(X_train, y_train)

pipeline = grid.best_estimator_
print("Best C:", grid.best_params_["model__C"])
print("Best CV ROC-AUC:", round(grid.best_score_, 4))

Best C: 10

Best CV ROC-AUC: 0.9638

In [55]:
# 11. Predict probability

y_probability = pipeline.predict_proba(X_test)[:, 1]

In [56]:
# 12. Predict Readmission

y_prediction = pipeline.predict(X_test)


In [57]:
# 13. ROC-AUC

auc = roc_auc_score(y_test, y_probability)

print("\nROC-AUC Score:", round(auc, 4))


ROC-AUC Score: 0.9497

In [58]:
# 14. Confusion Matrix

cm = confusion_matrix(y_test, y_prediction)

print("\nConfusion Matrix:")
print(cm)

Confusion Matrix:

[[120  24]

 [  1  52]]

In [59]:
# 15. Classification Report

print("\nClassification Report:")
print(classification_report(y_test, y_prediction))


Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.83      0.91       144

           1       0.68      0.98      0.81        53

    accuracy                           0.87       197

   macro avg       0.84      0.91      0.86       197

weighted avg       0.91      0.87      0.88       197

In [60]:
# 16. False Positive and False Negative

tn, fp, fn, tp = cm.ravel()

print("\nFalse Positive:", fp)
print("False Negative:", fn)
print("True Positive:", tp)
print("True Negative:", tn)

False Positive: 24

False Negative: 1

True Positive: 52

True Negative: 120

In [61]:
#16b. Threshold sensitivity:


from sklearn.metrics import precision_score, recall_score

for t in [0.5, 0.4, 0.3, 0.2]:
    preds = (y_probability >= t).astype(int)
    print(f"t={t} | recall={recall_score(y_test, preds):.2f} | precision={precision_score(y_test, preds):.2f}")

t=0.5 | recall=0.98 | precision=0.68

t=0.4 | recall=0.98 | precision=0.68

t=0.3 | recall=0.98 | precision=0.61

t=0.2 | recall=0.98 | precision=0.61

In [62]:
# 17. Clinical interpretation

print("\nClinical Interpretation:")

print("\nFalse Negative:")
print("The model predicts that the patient will NOT be readmitted,")
print("but the patient is actually readmitted.")

print("\nFalse Positive:")
print("The model predicts that the patient WILL be readmitted,")
print("but the patient is actually NOT readmitted.")

print("\nFalse negatives can be clinically important because")
print("a high-risk patient may not receive additional follow-up.")

Clinical Interpretation:

False Negative:

The model predicts that the patient will NOT be readmitted,

but the patient is actually readmitted.

False Positive:

The model predicts that the patient WILL be readmitted,

but the patient is actually NOT readmitted.

False negatives can be clinically important because

a high-risk patient may not receive additional follow-up.

In [63]:
# 18. Feature importance:

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
coefs = pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).drop(columns="abs_coef")

print(coef_df.head(10).to_string(index=False))

                                 feature      coef

 categorical__Procedure_X-Ray and Splint  3.137468

    categorical__Condition_Fractured Arm  3.137468

              categorical__Gender_Female  2.371939

                categorical__Gender_Male -2.364549

...

    categorical__Condition_Heart Disease  2.336435

                          numerical__Age  2.333259

  categorical__Condition_Prostate Cancer -1.922013

categorical__Procedure_Radiation Therapy -1.922013

   categorical__Condition_Osteoarthritis -1.879713